# 🚀 Marvedge Task-00046 — V3 Production Pipeline Benchmark
## Zero Transcode · MediaPipe BlazeFace · FFmpeg-Native Crop

> Connect with or without GPU — this pipeline is efficient either way.
> **Estimated time for a 20-min video: 4–8 minutes.**


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 1: System setup
# ─────────────────────────────────────────────────────────────────
import subprocess, os
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode == 0 else 'CPU only (still fast with V3)')

os.system('apt-get install -qq ffmpeg 2>/dev/null')
os.system('pip install -q mediapipe scenedetect[opencv] scipy tqdm opencv-python-headless')
print('✅ All dependencies installed')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 2: Clone repo (V3 pipeline is in this branch)
# ─────────────────────────────────────────────────────────────────
import os
BRANCH = 'feat/task-43-center-crop-fallback'
if not os.path.exists('/content/marvedge'):
    os.system(f'git clone -b {BRANCH} --depth 1 https://github.com/Marvedge/marvedge.git /content/marvedge')
else:
    os.system('git -C /content/marvedge pull')

os.chdir('/content/marvedge')
assert os.path.exists('scripts/ml/benchmark_preprocessing_v3.py'), 'V3 not found — pull may have failed'
print('✅ Repo ready — V3 pipeline confirmed')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 3: Mount Google Drive and locate video
# ─────────────────────────────────────────────────────────────────
from google.colab import drive
import os, subprocess

drive.mount('/content/drive', force_remount=True)

# ── List all video files in your Drive root to confirm the filename
print('\n📂 Video files found in MyDrive:')
for f in os.listdir('/content/drive/MyDrive/'):
    if any(f.lower().endswith(ext) for ext in ['.mp4', '.mkv', '.mov', '.avi']):
        size = os.path.getsize(f'/content/drive/MyDrive/{f}') / 1e9
        print(f'   → {f}  ({size:.2f} GB)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 4: Set VIDEO_PATH to your video
# ── EDIT the filename below to match what Cell 3 printed above ──
# ─────────────────────────────────────────────────────────────────
import os, subprocess

# ⬇️  UPDATE THIS LINE with the exact filename from Cell 3
VIDEO_FILENAME = 'kapil.mp4'

VIDEO_PATH = f'/content/drive/MyDrive/{VIDEO_FILENAME}'

assert os.path.lexists(VIDEO_PATH), f'File not found: {VIDEO_PATH}'

dur = float(subprocess.check_output(
    ['ffprobe','-v','error','-show_entries','format=duration',
     '-of','default=noprint_wrappers=1:nokey=1', VIDEO_PATH]
).decode().strip())
size_gb = os.path.getsize(VIDEO_PATH) / 1e9

print(f'✅ Video confirmed: {VIDEO_FILENAME}')
print(f'   Duration : {dur/60:.1f} min ({dur:.0f}s)')
print(f'   File size: {size_gb:.2f} GB')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 5: Run V3 Production Pipeline
# Reads directly from Drive — zero transcode, zero JPEG dump
# ─────────────────────────────────────────────────────────────────
import time, json, os

OUT_DIR = '/content/marvedge/demo/task46_v3'
REPORT  = f'{OUT_DIR}/benchmark_report.json'

print('=' * 65)
print('  🚀 V3 PRODUCTION PIPELINE')
print('  Zero transcode · MediaPipe BlazeFace · FFmpeg-native crop')
print('=' * 65)

t0 = time.time()
ret = os.system(
    f'python scripts/ml/benchmark_preprocessing_v3.py '
    f'--videoPath  "{VIDEO_PATH}" '
    f'--savePath   "{OUT_DIR}" '
    f'--reportPath "{REPORT}" '
    f'--stride 10 '
    f'--scale  0.5 '
    f'--threads 2'
)
t_total = time.time() - t0

if ret != 0:
    print('\n❌ Pipeline exited with error. Check output above.')
else:
    with open(REPORT) as f:
        rpt = json.load(f)
    print(f'\n✅ DONE in {t_total:.1f}s ({t_total/60:.1f} min)')
    print(f'   Throughput :  {rpt["fps_throughput"]} fps')
    print(f'   Realtime   :  {rpt["realtime_ratio"]}x')
    print(f'   Tracks     :  {rpt["tracks_found"]} ({rpt["fallback_tracks_found"]} fallback)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 6: Full report + download JSON
# ─────────────────────────────────────────────────────────────────
import json, datetime
from google.colab import files

with open(REPORT) as f:
    rpt = json.load(f)

print('╔══════════════════════════════════════════════════════════════╗')
print('║           V3 PRODUCTION PIPELINE — FINAL REPORT             ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Video      : {rpt["input_video"]:<47}║')
print(f'║  Duration   : {rpt["input_duration_sec"]/60:.1f} min{" "*44}║')
print(f'║  Resolution : {rpt["resolution"]:<47}║')
print(f'║  Detector   : MediaPipe BlazeFace (stride={rpt["detection_stride"]}, scale={rpt["detection_scale"]}x){" "*6}║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  {"STAGE":<33} {"TIME":>10}                  ║')
print('║  ' + '-'*58 + '║')
for s in rpt['stages']:
    print(f'║  {s["stage"]:<33} {s["wall_time_sec"]:>8.1f}s                 ║')
print('║  ' + '-'*58 + '║')
print(f'║  {"TOTAL":<33} {rpt["total_wall_time_sec"]:>8.1f}s                 ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Throughput : {rpt["fps_throughput"]} fps  ({rpt["realtime_ratio"]}x realtime){" "*18}║')
print(f'║  Tracks     : {rpt["tracks_found"]} ({rpt["fallback_tracks_found"]} fallback){" "*36}║')
print('╚══════════════════════════════════════════════════════════════╝')

# Save and download
out = '/content/task46_v3_results.json'
with open(out, 'w') as f:
    json.dump(rpt, f, indent=2)
files.download(out)
print('\n✅ Results downloaded as task46_v3_results.json')